In [ ]:
# This code is made to compare two models (YOLOv8n and custom best.pt) and to collect performance metrics
# Runs tracking on multiple video segments (different weather conditions)
# Shows for each model and segment: speed as FPS, inference time, objects detections per frame, detection confidence (averages),
# Shows number of cars with their max confidence captured during tracking for each segment
# counts IDs of detected cars for each segment as well as number of "ghosts" IDs - ids considered as incorrected - in this area there is need
# to make changes, it happens that a real car has several different IDs assigned or has ID wrong considered as incorret.
# Optionally you can save annotated output videos with bounding boxes and trajectories for visual verification



In [ ]:
# !pip install -q -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.7 MB/s eta 0:00:00


In [ ]:
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")
import torch
print(f"Torch version: {torch.__version__}")
import cv2
import numpy as np
from ultralytics import YOLO
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from torchvision import transforms
from IPython.display import display, clear_output
from collections import defaultdict



In [ ]:

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.rand(10000,10000).to(device)
print(x.device)

In [ ]:
!nvidia-smi

data\traffic.mp4
Detection_dev\model_training\yolo8_v0\main (1).ipynb

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [6]:
# Add or swap models here. Each entry needs:
MODELS = {
    "model_a": {
        "path":    "/content/best.pt",
        "imgsz":   416,
        "classes": [0], # custom single-class model
        "name":    "Custom YOLOv8 (416)",
    },
    "model_b": {
        "path":    "yolov8n.pt",
        "imgsz":   640,
        "classes": [2, 5, 7], # COCO: car, bus, truck
        "name":    "YOLOv8n COCO (640)",
    },
}


SEGMENTS = {
    "seg_sunny": {"path": "/content/seg_sunny.mp4", "label": "Sunny"},
    "seg_fog":   {"path": "/content/seg_fog.mp4",   "label": "Fog"},
    "seg_rain":  {"path": "/content/seg_rain.mp4",  "label": "Rain"},
    "seg_night": {"path": "/content/seg_night.mp4", "label": "Night"},
}

# choose output videos to save
# (model_key, segment_key)
# To save all 8 set SAVE_VIDEOS = "all", none = []

SAVE_VIDEOS = [
    ("model_a", "seg_fog"),
    ("model_b", "seg_fog"),
]

CONF_THRESHOLD = 0.3
TRACKER = "bytetrack.yaml"
TRACK_COLOR = (0, 255, 0)
BOUNDING_BOX_COLOR = (255, 255, 255)
LINE_THICKNESS = 1
VIDEO_FOURCC = "mp4v"

# iDs seen fewer times than this are counted as ghosts, need to be changed
GHOST_THRESHOLD = 5

In [7]:
class Car:
    def __init__(self, trackId: int, frame_index: int):
        self.trackId = trackId
        self.x = 0.0
        self.y = 0.0
        self.w = 0.0
        self.h = 0.0
        self.maxConfidence = 0.0
        self.last_confidence = 0.0
        self.label = ""
        self.history = []
        self.lastCrop = None
        self.last_seen = frame_index
        self.first_seen = frame_index # needed for lifetime calculation
        self.updateCount = 0
        self.type = "unknown"
        # self.Deaccelaration = self.getDeceleration()
        # self.speed = 0.0
        # self.breakingDistance = self.calcBreakingDistance()

    def calcBreakingDistance(self):
        # Placeholder logic for calculating breaking distance
        pass


    # def  getDeceleration(self):
    #     type = self.type
    #     array = {
    #         "sedan": 3.0,
    #         "category2": 2.0,
    #         "bus": 2.5,
    #         "motorcycle": 4.0,
    #         "van": 2.5,
    #         "unknown": 2.0}

    #     return array.get(type, 2.0)


    def checkType(self):
        # Placeholder classification logic
        return "sedan"

    def update(self, box: tuple, confidence: float, label: str, frame: np.ndarray, frame_index: int) -> None:
        self.updateCount    += 1
        self.x, self.y, self.w, self.h = box
        self.label           = label
        self.last_confidence = confidence
        self.last_seen       = frame_index
        self.history.append((float(self.x), float(self.y)))

        if confidence > self.maxConfidence:
            self.maxConfidence = confidence
            x1 = int(self.x - self.w / 2)
            y1 = int(self.y - self.h / 2)
            x2 = int(self.x + self.w / 2)
            y2 = int(self.y + self.h / 2)
            crop = frame[max(0, y1):min(frame.shape[0], y2), max(0, x1):min(frame.shape[1], x2)]
            if crop.size > 0:
                self.lastCrop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                # self.type = self.checkType()


In [8]:

def draw_custom_box(annotatedFrame, box_xyxy, trackId, conf, BOUNDING_BOX_COLOR, LINE_THICKNESS,font_scale=0.4, font_thickness=1):
    x1, y1, x2, y2 = map(int, box_xyxy)
    label_text = f"car {trackId} | {conf:.2f}"

    cv2.rectangle(annotatedFrame, (x1, y1), (x2, y2), BOUNDING_BOX_COLOR, LINE_THICKNESS)

    (text_width, text_height), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)
    cv2.rectangle(annotatedFrame, (x1, y1 - text_height - 5), (x1 + text_width, y1), BOUNDING_BOX_COLOR, -1)
    cv2.putText(annotatedFrame, label_text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), font_thickness)

In [9]:
# Benchmark and tracking function, one model, one segment

def run_benchmark(model_key: str, segment_key: str, save_video: bool) -> dict | None:

    cfg_m = MODELS[model_key]
    cfg_s = SEGMENTS[segment_key]

    if not os.path.exists(cfg_s["path"]):
        print(f" File not found: {cfg_s['path']}")
        return None

    print(f"\n>  {cfg_m['name']}  |  {cfg_s['label']}")

    model = YOLO(cfg_m["path"])
    cap = cv2.VideoCapture(cfg_s["path"])
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))


    out = None
    if save_video:
        frameWidth = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frameHeight = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        output_path = f"/content/output_{model_key}_{segment_key}.mp4"
        fourcc = cv2.VideoWriter_fourcc(*VIDEO_FOURCC)
        out = cv2.VideoWriter(output_path, fourcc, fps, (frameWidth, frameHeight))
        print(f"   Saving video: {output_path}")

    # lists, values per frame
    frame_fps = []
    frame_inference_ms = []
    frame_detections = []   # list[int]
    frame_confidences = []   # list[float}

    carsDict = {}   # dict[int, Car]
    frame_index = 0

    try:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            results = model.track(
                source = frame,
                imgsz = cfg_m["imgsz"],
                conf = CONF_THRESHOLD,
                persist = True,
                verbose = False,
                device = 0 if device == "cuda" else "cpu",
                tracker = TRACKER,
                classes = cfg_m["classes"],
            )

            # speed
            spd = results[0].speed  # preprocess/inference/postprocess [ms]
            total_ms = spd["preprocess"] + spd["inference"] + spd["postprocess"]
            frame_fps.append(1000.0 / total_ms if total_ms > 0 else 0.0)
            frame_inference_ms.append(spd["inference"])

            # detect
            annotatedFrame = frame.copy()

            if results[0].boxes.id is not None:
                boxes_xyxy = results[0].boxes.xyxy.cpu().numpy()
                boxes_xywh = results[0].boxes.xywh.cpu().numpy()
                trackIds = results[0].boxes.id.int().cpu().tolist()
                classIndices = results[0].boxes.cls.int().cpu().tolist()
                confidences = results[0].boxes.conf.cpu().tolist()

                frame_detections.append(len(trackIds))
                frame_confidences.append(float(np.mean(confidences)))

                for box_xyxy, box_xywh, trackId, classIdx, conf in zip(boxes_xyxy, boxes_xywh, trackIds, classIndices, confidences):

                    original_label = model.names[classIdx]

                    if trackId not in carsDict:
                        carsDict[trackId] = Car(trackId, frame_index)

                    car = carsDict[trackId]
                    car.update(box_xywh, conf, original_label, frame, frame_index)

                    if out is not None:
                        draw_custom_box(annotatedFrame, box_xyxy, trackId, conf, BOUNDING_BOX_COLOR, LINE_THICKNESS)
                        points = np.array(car.history).astype(np.int32).reshape((-1, 1, 2))
                        cv2.polylines(annotatedFrame, [points], isClosed=False, color=TRACK_COLOR, thickness=LINE_THICKNESS)
                        # notice there is no deleting cars from carsDict!!
            else:
                frame_detections.append(0)
                frame_confidences.append(0.0)

            if out is not None:
                out.write(annotatedFrame)

            frame_index += 1
            if frame_index % 50 == 0:
                print(f"  frame {frame_index}/{total_frames}  |  "f"FPS: {frame_fps[-1]:.1f}  |  "f"detections: {frame_detections[-1]}")

    except KeyboardInterrupt:
        print(" Interrupted")
    finally:
        cap.release()
        if out is not None:
            out.release()

    # post loop objects lists, need to be fixed, real cars and ghost cars have to be determined in other way, to do in future
    all_ids = list(carsDict.values()) # list of each object in carsDict
    real_cars = [c for c in all_ids if c.updateCount >= GHOST_THRESHOLD] # this threshold needs to be changed
    ghost_cars = [c for c in all_ids if c.updateCount <  GHOST_THRESHOLD]
    id_lifetimes = [c.last_seen - c.first_seen for c in real_cars]
    max_confs = [c.maxConfidence for c in real_cars]


    print(f"\n  --- ID summary ({cfg_s['label']}) ---")
    print(f"  Real cars  (>= {GHOST_THRESHOLD} updates): {len(real_cars)}")
    for c in sorted(real_cars, key=lambda c: c.trackId):
        print(f"    ID {c.trackId:>4}  updates: {c.updateCount:>5}  "
              f"max_conf: {c.maxConfidence:.2f}  "
              f"frames alive: {c.last_seen - c.first_seen}")
    print(f"  Ghosts (< {GHOST_THRESHOLD} updates): {len(ghost_cars)}")

    return {
        "model_key":     model_key,
        "model_name":    cfg_m["name"],
        "segment_key":   segment_key,
        "segment_label": cfg_s["label"],

        "mean_fps":          np.mean(frame_fps),
        "mean_inference_ms": np.mean(frame_inference_ms),

        "mean_detections":       np.mean(frame_detections),
        "zero_detection_frames": int(np.sum(np.array(frame_detections) == 0)),
        "total_frames":          frame_index,
        "mean_confidence":       np.mean([c for c in frame_confidences if c > 0])
                                 if any(c > 0 for c in frame_confidences) else 0.0,

        "unique_ids":              len(all_ids),
        "real_car_ids":            len(real_cars),
        "ghost_ids":               len(ghost_cars),
        "mean_id_lifetime_frames": np.mean(id_lifetimes) if id_lifetimes else 0.0,
        "mean_max_confidence":     np.mean(max_confs)    if max_confs    else 0.0,

        "_frame_fps":         frame_fps,
        "_frame_detections":  frame_detections,
        "_frame_confidences": frame_confidences,

        "_max_confs":    max_confs,
        "_id_lifetimes": id_lifetimes,
    }

In [ ]:
# Runing benchmark for every model and segment combination

if SAVE_VIDEOS == "all":
    _save_set = {(mk, sk) for mk in MODELS for sk in SEGMENTS}
else:
    _save_set = set(SAVE_VIDEOS)

all_results = []

for model_key in MODELS:
    for segment_key in SEGMENTS:
        result = run_benchmark(
            model_key,
            segment_key,
            save_video=(model_key, segment_key) in _save_set,
        )
        if result is not None:
            all_results.append(result)

print("\n Benchmark complete.")


In [ ]:
# Summary charts

segment_keys = [sk for sk in SEGMENTS
                  if any(r["segment_key"] == sk for r in all_results)]
segment_labels = [SEGMENTS[sk]["label"] for sk in segment_keys]
model_keys_run = [mk for mk in MODELS
                  if any(r["model_key"] == mk for r in all_results)]
model_names = [MODELS[mk]["name"] for mk in model_keys_run]
colors = plt.cm.tab10.colors

x = np.arange(len(segment_labels))
bar_w = 0.8 / len(model_keys_run)

def get_val(mk, sk, metric):
    for r in all_results:
        if r["model_key"] == mk and r["segment_key"] == sk:
            return r.get(metric, 0)
    return 0

def draw_bars(ax, metric, ylabel, title):
    for i, (mk, mn) in enumerate(zip(model_keys_run, model_names)):
        vals   = [get_val(mk, sk, metric) for sk in segment_keys]
        offset = (i - len(model_keys_run) / 2 + 0.5) * bar_w
        ax.bar(x + offset, vals, bar_w, label=mn, color=colors[i])
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(segment_labels)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.4)

fig, axes = plt.subplots(3, 2, figsize=(16, 13))
fig.suptitle("YOLOv8 Tracking Benchmark – Summary", fontsize=14, fontweight="bold")
plt.subplots_adjust(hspace=0.45, wspace=0.35)

draw_bars(axes[0, 0], "mean_fps",             "FPS",              "Average FPS")
draw_bars(axes[0, 1], "mean_inference_ms",    "ms",               "Average Inference Time [ms]")
draw_bars(axes[1, 0], "mean_detections",      "objects / frame",  "Average Detections per Frame")
draw_bars(axes[1, 1], "zero_detection_frames","frames",           "Frames with Zero Detections")
draw_bars(axes[2, 1], "mean_confidence",      "confidence [0-1]", "Average Detection Confidence")
axes[2, 1].set_ylim(0, 1)

ax5 = axes[2, 0]
for i, (mk, mn) in enumerate(zip(model_keys_run, model_names)):
    real  = [get_val(mk, sk, "real_car_ids") for sk in segment_keys]
    ghost = [get_val(mk, sk, "ghost_ids")    for sk in segment_keys]
    offset = (i - len(model_keys_run) / 2 + 0.5) * bar_w
    ax5.bar(x + offset, real,  bar_w,
            label=f"{mn} – real",   color=colors[i], alpha=0.9)
    ax5.bar(x + offset, ghost, bar_w,
            label=f"{mn} – ghosts", color=colors[i], alpha=0.4,
            hatch="//", bottom=real)
ax5.set_title(f"Unique IDs: real (>={GHOST_THRESHOLD} upd.) vs ghosts")
ax5.set_xticks(x)
ax5.set_xticklabels(segment_labels)
ax5.set_ylabel("ID count")
ax5.legend(fontsize=7)
ax5.grid(axis="y", alpha=0.4)

plt.savefig("/content/metrics_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /content/metrics_summary.png")


In [ ]:
# Max confidence histograms

n_rows = len(model_keys_run)
n_cols = len(segment_keys)

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(5 * n_cols, 4.5 * n_rows),
                         sharey=False)
fig.suptitle(
    f"Max Confidence per Car – real IDs (>={GHOST_THRESHOLD} updates)",
    fontsize=13, fontweight="bold")

if n_rows == 1:
    axes = [axes]
if n_cols == 1:
    axes = [[ax] for ax in axes]

for row, (mk, mn) in enumerate(zip(model_keys_run, model_names)):
    for col, sk in enumerate(segment_keys):
        ax = axes[row][col]
        r  = next((r for r in all_results
                   if r["model_key"] == mk and r["segment_key"] == sk), None)

        if r and r["_max_confs"]:
            ax.hist(r["_max_confs"], bins=20,
                    color=colors[row], alpha=0.85,
                    edgecolor="white", linewidth=0.5)
            mean_val = np.mean(r["_max_confs"])
            ax.axvline(mean_val, color="red", linestyle="--",
                       linewidth=1.2, label=f"mean {mean_val:.2f}")
            ax.legend(fontsize=8)
        else:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes, color="grey")

        ax.set_title(f"{mn}\n{SEGMENTS[sk]['label']}", fontsize=9)
        ax.set_xlabel("max confidence")
        ax.set_ylabel("number of cars")
        ax.set_xlim(0, 1)
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/content/metrics_conf_histograms.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /content/metrics_conf_histograms.png")